# CSCI 447/547 Hackathon 9: Recommender Systems

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will build several movie recommenders, from a simple list of popular movies to collaborative filtering and content-based filtering, and evaluate them on the MovieLens dataset.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon9-recommender_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [`u.data` and `u.item` from GitHub](https://github.com/lucywowen/csci547_ML/tree/main/examples/data/recommender) and upload them to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon9-recommender_systems.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

---

## 1. Recommender Systems

Every time Netflix suggests a show, Spotify builds a playlist, or Amazon tells you that "customers who bought this also bought...", a **recommender system** is predicting what you'll like. There are two main approaches:

- **Content-based filtering** recommends items that are *similar* to ones you liked, based on the items' attributes (for movies: genre, director, actors, ...).
- **Collaborative filtering (CF)** uses the "wisdom of the crowd": it recommends what *similar users* liked, using only the ratings, without knowing anything about the items themselves. **Memory-based** CF compares users (or items) directly through their ratings, while **model-based** CF learns a compact model of all of the ratings.

### Think-Pair-Share 1: The Cold-Start Problem
**Context:** A recommender is only as good as the information it has.

1. **Think:** A brand-new movie is added to the catalog, with no ratings yet. Which approach can recommend it, and why? What about a brand-new user who hasn't rated anything?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Besides star ratings, what other information could a service like Netflix use to learn what you like? Which is more plentiful?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 1.1 Load the Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid')

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse.linalg import svds

### 1.2 Load the Data

We'll use **MovieLens 100K**, a classic recommender dataset collected by the GroupLens research lab at the University of Minnesota: 100,000 ratings (from 1 to 5 stars) of 1,682 movies by 944 users. `u.data` holds the ratings, and `u.item` holds each movie's title and which of 19 genres it belongs to. (Depending on your environment/data location this cell may need to change.)

In [ ]:
genre_names = ['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
               'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movie_columns = ['item_id', 'title', 'release_date', 'video_release_date', 'url'] + genre_names

try:
    from google.colab import files
    uploaded = files.upload()

    ratings = pd.read_csv('u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']) # colab file import
    movies = pd.read_csv('u.item', sep='|', encoding='latin-1', names=movie_columns)

except:

    ratings = pd.read_csv('data/recommender/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']) # local file import
    movies = pd.read_csv('data/recommender/u.item', sep='|', encoding='latin-1', names=movie_columns)

# one text column listing each movie's genres, like "Animation|Children's|Comedy"
movies['genres'] = movies[genre_names].apply(lambda row: '|'.join(g for g in genre_names if row[g] == 1), axis=1)
movies = movies[['item_id', 'title', 'genres']]

print(f"{len(ratings)} ratings of {ratings['item_id'].nunique()} movies by {ratings['user_id'].nunique()} users")
ratings.head()

### 1.3 Coding Exercise: The Simplest Recommender

Before we get personal, let's build the simplest recommender of all: recommend the same highly rated movies to everyone.

**What you need to write:** Make a table of the 10 movies with the highest average rating, among the movies with **at least 100 ratings**, showing each movie's average rating and its number of ratings. `ratings_with_titles` (handled for you) adds each movie's title to its ratings. `groupby()` (from Hackathon 7) can group the ratings by title, and `agg()` computes several summaries at once: for example, `df.groupby('a')['b'].agg(['mean', 'count'])` gives the mean and the count of column `b` for each value of `a`. (Why do you think we require at least 100 ratings?)

*Expected:* A Close Shave (1995) at the top, with an average of 4.49 from 112 ratings, followed by Schindler's List and The Wrong Trousers.

In [ ]:
ratings_with_titles = ratings.merge(movies, on='item_id') # (handled for you)

# --- YOUR CODE HERE ---

## 2. The User-Item Matrix

Collaborative filtering arranges the ratings in a **user-item matrix**, with one row per user and one column per movie. Most users have only rated a few dozen of the 1,682 movies, so the matrix is mostly empty: it's **sparse**.

To see whether we can predict ratings we haven't seen, we hide 40% of the ratings, just like the train/fine-tune/generalization splits in earlier hackathons: 20% become the **fine-tuning** set (for choosing between models) and 20% become the **test** set (touched once, at the very end). Then we build one user-item matrix for each set (handled for you), storing missing ratings as 0.

In [ ]:
n_users = ratings['user_id'].max() + 1 # user IDs start at 0, so they can be used as row numbers
n_movies = ratings['item_id'].max()    # movie IDs start at 1, so movie i is in column i - 1

train, temp = train_test_split(ratings, test_size=0.4, random_state=42)
fine, test = train_test_split(temp, test_size=0.5, random_state=42)

def user_item_matrix(df):
    matrix = np.zeros((n_users, n_movies))
    matrix[df['user_id'], df['item_id'] - 1] = df['rating']
    return matrix

R_train, R_fine, R_test = user_item_matrix(train), user_item_matrix(fine), user_item_matrix(test)

print('User-item matrix shape:', R_train.shape)
print(f'Sparsity: {1 - len(ratings) / (n_users * n_movies):.1%} of the matrix is empty')

### Think-Pair-Share 2: What Does a Missing Rating Mean?
**Context:** About 94% of the user-item matrix is empty, and we store every missing rating as a 0.

1. **Think:** Why would it be a mistake for a recommender to treat those 0s as if users had rated the movies 0 stars?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** People choose which movies to watch (and rate). Do you think the missing ratings are missing *at random*? What does that mean for a recommender?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 2.1 A Baseline

To judge our recommenders, we'll use the **root mean squared error (RMSE)** between the predicted and actual ratings, over the ratings in the fine-tuning set only (the empty cells are skipped). An RMSE of 1 means our predictions are off by about 1 star. Our baseline "model" predicts every movie at the user's average rating (handled for you):

In [ ]:
def rmse(predictions, truth):
    has_rating = truth > 0 # only score the ratings that actually exist
    return np.sqrt(np.mean((predictions[has_rating] - truth[has_rating])**2))

rated = R_train > 0
user_mean = R_train.sum(axis=1) / np.maximum(rated.sum(axis=1), 1) # each user's average rating (ignoring the 0s)
user_mean[rated.sum(axis=1) == 0] = R_train[rated].mean()           # users with no training ratings get the overall average

baseline = np.tile(user_mean[:, np.newaxis], (1, n_movies)) # every movie predicted at the user's average
print(f"Baseline (each user's average) RMSE: {rmse(baseline, R_fine):.3f}")

## 3. Memory-Based Collaborative Filtering

**User-based** collaborative filtering predicts how you'll rate a movie from how *similar users* rated it: "users who are similar to you also liked...". To measure how similar two users are, we treat their rows of the user-item matrix as vectors and use the **cosine similarity**, the cosine of the angle between them:

$$\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\Vert\mathbf{r}_u\Vert \, \Vert\mathbf{r}_v\Vert}$$

Users who rate the same movies in similar ways point in similar directions, so their similarity is close to 1. Users with nothing in common have a similarity of 0.

### Think-Pair-Share 3: Different Rating Scales
**Context:** User A gives their favorite movies 4 stars and everything else 3 stars. User B gives their favorites 5 stars and the movies they fell asleep during 3 stars.

1. **Think:** Could users A and B have the same taste? If user B rates a new movie 5 stars, what rating would you predict for user A?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** How could we adjust the ratings so that users with different rating habits can be compared fairly?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

That's exactly how our user-based recommender works. Its predicted rating for user $u$ and movie $m$ is user $u$'s average, plus a similarity-weighted average of how far above or below their own averages the users who rated $m$ rated it (handled for you):

In [ ]:
user_similarity = cosine_similarity(R_train) # (944 x 944): the similarity of every pair of users

def predict_user_based(R, similarity):
    rated = R > 0
    deviations = (R - user_mean[:, np.newaxis]) * rated # how far above or below their own average each user rated each movie
    weighted_sum = similarity @ deviations               # similarity-weighted sum of the deviations, for every user and movie
    total_weight = np.abs(similarity) @ rated            # total similarity of the users who rated each movie
    adjustment = np.divide(weighted_sum, total_weight, out=np.zeros_like(weighted_sum), where=total_weight > 0)
    return user_mean[:, np.newaxis] + adjustment

user_based = predict_user_based(R_train, user_similarity)
print(f'User-based collaborative filtering RMSE: {rmse(user_based, R_fine):.3f}')

### 3.1 Coding Exercise: "Users Who Liked This Also Liked..."

**Item-based** collaborative filtering flips this around: two *movies* are similar if the same users rated them in similar ways. Their similarity is the cosine similarity between their *columns* of the user-item matrix (handled for you, in the first line of the cell below).

**What you need to write:** Write a function `most_similar(similarity, movie, n=10)` that returns the titles of the `n` movies that are most similar to the movie in row `movie` of a movie-by-movie `similarity` matrix, most similar first. Every movie is most similar to itself, so leave the movie itself out. `np.argsort()` sorts the positions of an array's values from smallest to largest (we reversed it with `[::-1]` to order the eigenvalues in Hackathon 8), and `movies['title'].iloc[positions]` looks up the titles of an array of row positions. The helper `find_movie()` (handled for you) gives a movie's row from its title.

*Expected:* Return of the Jedi (1983) first, followed by Toy Story (1995), Twelve Monkeys (1995), and The Empire Strikes Back (1980).

In [ ]:
movie_similarity = cosine_similarity(R_train.T) # (1682 x 1682): the similarity of every pair of movies

def find_movie(title):
    """Returns the row (and column) number of the movie with this title."""
    return movies.index[movies['title'] == title][0]


def most_similar(similarity, movie, n=10):
    """Returns the titles of the n movies most similar to the movie in row `movie` of `similarity`."""
    # --- YOUR CODE HERE ---


most_similar(movie_similarity, find_movie('Star Wars (1977)'))

## 4. Model-Based Collaborative Filtering: Matrix Factorization

Memory-based CF needs the similarity between *every* pair of users, which gets very expensive with millions of users. **Matrix factorization** compresses the ratings instead. It approximates the user-item matrix as the product of two much smaller matrices: one describing each user with $k$ numbers, and one describing each movie with the same $k$ numbers.

<img alt="matrix factorization of a user-item matrix" src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*F5RM9oMnJcWi1oxaEhF0mg.png" width=650>

These $k$ numbers are **latent factors**: hidden features learned from the ratings alone, like "how much of an action movie is this?" and "how much does this user like action movies?". A predicted rating is the dot product of the user's and the movie's factors, so a user is predicted to like movies whose factors line up with their own.

We'll use the **singular value decomposition (SVD)**, the same method `scikit-learn`'s `PCA` used under the hood in Hackathon 8: SVD finds the $k$ factors that reconstruct the matrix best. Following Think-Pair-Shares 2 and 3, we factorize each user's mean-centered ratings, with missing ratings set to 0 (the user's average), then add each user's average back. Here it is with $k = 20$ (handled for you). `svds()` returns the user factors, the strength of each factor, and the movie factors:

In [ ]:
deviations = (R_train - user_mean[:, np.newaxis]) * (R_train > 0) # mean-centered ratings, with missing ratings at 0

user_factors, strengths, movie_factors = svds(deviations, k=20)
svd_predictions = user_mean[:, np.newaxis] + user_factors @ np.diag(strengths) @ movie_factors
print(f'SVD (k = 20) RMSE: {rmse(svd_predictions, R_fine):.3f}')

### 4.1 Coding Exercise: How Many Latent Factors?

$k$ is a hyperparameter: too few factors can't capture different tastes, and too many start fitting noise.

**What you need to write:** By the end of the loop, `svd_rmse` should hold the fine-tuning RMSE of the SVD recommender above for each number of factors in `k_values`. Then plot `svd_rmse` against `k_values`, with a dashed horizontal line at the baseline's RMSE for comparison, labeled so the legend can tell them apart.

*Expected:* the lowest RMSE (about 1.004) is at $k = 10$, and the RMSE climbs to about 1.033 at $k = 100$.

In [ ]:
k_values = [2, 5, 10, 20, 50, 100]
svd_rmse = []

for k in k_values:
    # --- YOUR CODE HERE ---

plt.figure(figsize=(8, 5))
# --- YOUR CODE HERE ---
plt.xlabel('number of latent factors (k)')
plt.ylabel('fine-tuning RMSE')
plt.title('Choosing the Number of Latent Factors')
plt.legend()
plt.show()

### 4.2 Evaluate on the Test Set

Now that we've made our choices with the fine-tuning set, let's score each recommender once on the test set (handled for you):

In [ ]:
user_factors, strengths, movie_factors = svds(deviations, k=10)
svd_predictions = user_mean[:, np.newaxis] + user_factors @ np.diag(strengths) @ movie_factors

print(f"Baseline (each user's average): {rmse(baseline, R_test):.3f}")
print(f'User-based CF:                  {rmse(user_based, R_test):.3f}')
print(f'SVD (k = 10):                   {rmse(svd_predictions, R_test):.3f}')

Is a drop from about 1.04 to 0.96 a big deal? In 2006, Netflix offered **$1 million** to the first team that could beat its own recommender's RMSE by 10%. It took three years! The winning teams relied heavily on matrix factorization, but trained only on the ratings that exist, rather than filling in the missing ones like our simple version, plus many more tricks.

**Pause point:** Our simple user-based recommender beat SVD. Look back at how we handled the missing ratings before factorizing: every empty cell became "exactly this user's average". How might pretending to know 94% of the ratings hold the SVD back?

## 5. Content-Based Filtering

A content-based recommender ignores the ratings entirely and recommends movies with similar *attributes*: here, their genres. To turn genres into vectors, we use `TfidfVectorizer`, which works like the `CountVectorizer` from Hackathon 4 but gives **rare** genres more weight than common ones. Sharing "Film-Noir" with another movie says more than sharing "Drama", since about 40% of the movies are dramas. Here are the genre vectors (handled for you):

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r'[^|]+') # each genre between the | characters is one "word"
genre_vectors = tfidf.fit_transform(movies['genres']) # (1682 x 19): one row per movie, one column per genre

pd.DataFrame(genre_vectors.toarray(), columns=tfidf.get_feature_names_out(), index=movies['title']).loc[['Star Wars (1977)', 'Toy Story (1995)']].round(2)

### 5.1 Coding Exercise: Collaborative vs. Content-Based

**What you need to write:** Compute `genre_similarity`, the cosine similarity between every pair of movies' genre vectors (just like `movie_similarity` in Section 3.1, but from `genre_vectors`). Then use your `most_similar()` function to display a table of the 10 movies most similar to **Toy Story (1995)**, with one column for collaborative filtering and one for content-based filtering. `pd.DataFrame()` can build the table from a dictionary of lists: `pd.DataFrame({'column name': a_list, ...})`.

*Expected:* the content-based column starts with Aladdin and the King of Thieves (1996) and Balto (1995), while the collaborative filtering column starts with Star Wars (1977) and The Rock (1996).

In [ ]:
# --- YOUR CODE HERE ---

### Think-Pair-Share 4: Which List Would You Rather Get?
**Context:** Compare the two columns of your table, and look back at the Star Wars list from Section 3.1.

1. **Think:** Describe what each method recommends for a Toy Story fan. What is each one good at, and what does each one miss?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** How could you combine the two approaches to get the best of both?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

### Comparing the Approaches

| | Needs | Handles new movies? | Explainable? | Scales to millions of users? |
| :--- | :--- | :--- | :--- | :--- |
| **Memory-based CF** | ratings only | no | yes ("users like you liked it") | poorly (compares every pair) |
| **Matrix factorization** | ratings only | no | hardly (latent factors) | yes (a few numbers per user and movie) |
| **Content-based** | item attributes | yes | yes ("it's also an animated comedy") | yes |

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
Our user-based recommender needed the similarity between every pair of our 944 users. Netflix has hundreds of millions of users. Why does that make memory-based collaborative filtering impractical at Netflix's scale, and how does matrix factorization avoid the problem? What does memory-based CF offer in return?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate's recommender keeps suggesting obscure movies that only one or two people have rated, all with 5 stars. What's going on, and how could they fix it?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world recommender you use, and whether you think it's content-based, collaborative, or a hybrid:

    1. 